[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/ch11_fundamental_theorem.ipynb)

# Chapter 11 — Companion Notebook
## The Fundamental Theorem of Calculus
*why finding an area is the same as undoing a derivative — the summit of the book*

Four live pictures of the chapter's one idea and its payoff: slide the right edge of an area and watch its growth rate land exactly on the curve (Part 1); turn a definite integral into a single subtraction $F(b)-F(a)$ (Part 2); run the product rule backward with integration by parts; and, as the book's final act, watch $\ln x$ appear as the area under $1/t$ and find $e$ as the place that area first hits $1$.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. The moving edge — FTC Part 1

Fix the left edge at $a$ and let the **right edge** $x$ slide. Define the *area-so-far function*

$$A(x) = \int_a^x f(t)\,dt,$$

the signed area under $f$ from $a$ out to $x$. The whole theorem is the answer to one question: *how fast does this area grow as $x$ moves right?* Push the edge out by a sliver $h$ and the area gains a thin strip of width $h$ and height about $f(x)$ — so the growth rate is the height of the curve. In symbols, **Part 1**:

$$A'(x) = f(x).$$

Pick a curve and drag $x$. The **top** panel shades $A(x)$ in green and shows the strip being added at the right edge. The **bottom** panel plots the accumulated area $A(x)$ in red, and overlays its *numerical* derivative $A'(x)$ as a green dashed line — watch it land exactly on the blue dot $f(x)$. The Fundamental Theorem, seen live — *(Figure 11.1)*.

In [2]:
_CURVES1 = {
    'f(t) = sin(t) + 1.5':   (lambda t: np.sin(t) + 1.5,  0.0, 6.2),
    'f(t) = t^2':            (lambda t: t**2,             0.0, 2.4),
    'f(t) = 0.5 t + 0.5':    (lambda t: 0.5*t + 0.5,      0.0, 4.0),
    'f(t) = e^{t/3}':        (lambda t: np.exp(t/3),      0.0, 4.0),
    'f(t) = 4 - (t-2)^2':    (lambda t: 4 - (t-2)**2,     0.0, 4.0),
}

def _area_to(f, a, x, n=600):
    '''Numerical signed area A(x) = int_a^x f, via the trapezoid rule.'''
    if x <= a:
        return 0.0
    ts = np.linspace(a, x, n)
    return float(np.trapz(f(ts), ts))

def _moving_edge(curve='f(t) = sin(t) + 1.5', x=3.0):
    f, a, b = _CURVES1[curve]
    x = min(max(x, a + 0.05), b)            # keep x inside [a, b]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 6.8),
                                   sharex=True)
    ts = np.linspace(a, b, 600)
    ys = f(ts)

    # --- Top: the curve and the shaded area A(x). ---
    ax1.plot(ts, ys, color=BLUE, lw=2.4, label='$f(t)$', zorder=3)
    mask = ts <= x
    ax1.fill_between(ts[mask], 0, ys[mask], color=SHADEG,
                     alpha=0.95, zorder=1,
                     label='area $A(x)=\\int_a^x f$')
    # The thin strip added at the right edge.
    h = 0.10 * (b - a)
    xs = np.linspace(x, min(x + h, b), 30)
    ax1.fill_between(xs, 0, f(xs), color=RED, alpha=0.45, zorder=2,
                     label='new strip ($\\approx f(x)\\,h$)')
    ax1.axvline(x, color=RED, lw=1.2, ls=':')
    ax1.scatter([x], [f(x)], s=70, color=BLUE, zorder=6)
    ax1.axhline(0, color=GRAY, lw=0.8)
    ax1.set_ylabel('$f(t)$')
    ax1.set_title(f'Area so far:  $A(x) \\approx$ {_area_to(f, a, x):.3f}',
                  color=DARK)
    ax1.legend(loc='upper left', framealpha=0.9, fontsize=9)

    # --- Bottom: A(x) and its numerical derivative vs f. ---
    xx = np.linspace(a, b, 200)
    Avals = np.array([_area_to(f, a, t) for t in xx])
    dA = np.gradient(Avals, xx)            # numerical A'(x)
    ax2.plot(xx, Avals, color=RED, lw=2.4, label='$A(x)$', zorder=3)
    ax2.plot(xx, dA, color=GREEN, lw=2.2, ls='--',
             label="$A'(x)$ (numerical)", zorder=4)
    ax2.plot(ts, ys, color=BLUE, lw=1.4, alpha=0.55,
             label='$f(t)$', zorder=2)
    ax2.scatter([x], [f(x)], s=70, color=BLUE, zorder=6)
    ax2.scatter([x], [_area_to(f, a, x)], s=70, color=RED, zorder=6)
    ax2.axvline(x, color=RED, lw=1.2, ls=':')
    ax2.axhline(0, color=GRAY, lw=0.8)
    ax2.set_xlabel('$x$')
    ax2.set_title("$A'(x)$ (green dashed) lands on $f(x)$ (blue) "
                  "\u2014 Part 1", color=DARK)
    ax2.legend(loc='upper left', framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()

interact(_moving_edge,
         curve=Dropdown(options=list(_CURVES1), description='curve'),
         x=FloatSlider(value=3.0, min=0.1, max=6.2, step=0.1,
                       description='right edge x'));


interactive(children=(Dropdown(description='curve', options=('f(t) = sin(t) + 1.5', 'f(t) = t^2', 'f(t) = 0.5 …

## 2. Area becomes a subtraction — FTC Part 2

Since $A(x)=\int_a^x f$ is *an* antiderivative of $f$ (Part 1), and any other antiderivative $F$ differs from it only by a constant, the laborious definite integral collapses into arithmetic. **Part 2:**

$$\int_a^b f(x)\,dx = F(b) - F(a) = \big[F(x)\big]_a^b.$$

Find an antiderivative, plug in the endpoints, subtract. Pick a pair $(f, F)$ and drag the endpoints $a$ and $b$. The shaded green area under $f$ (top) equals the **rise** of $F$ between the two marked endpoint values $F(a)$ and $F(b)$ (bottom). Read the title: the trapezoid-rule area and $F(b)-F(a)$ agree to the decimal.

In [3]:
# Each preset: (f, F) with F' = f, plus a default [a, b] window.
_PAIRS = {
    'f=x^2,  F=x^3/3':
        (lambda x: x**2, lambda x: x**3/3, -1.5, 2.0),
    'f=sin x,  F=-cos x':
        (lambda x: np.sin(x), lambda x: -np.cos(x), 0.0, np.pi),
    'f=cos x,  F=sin x':
        (lambda x: np.cos(x), lambda x: np.sin(x), 0.0, np.pi/2),
    'f=e^x,  F=e^x':
        (lambda x: np.exp(x), lambda x: np.exp(x), 0.0, 1.5),
    'f=1/x,  F=ln x':
        (lambda x: 1.0/x, lambda x: np.log(x), 1.0, 3.0),
    'f=2x+1,  F=x^2+x':
        (lambda x: 2*x + 1, lambda x: x**2 + x, -1.0, 2.0),
}

def _trap_area(f, a, b, n=800):
    ts = np.linspace(a, b, n)
    return float(np.trapz(f(ts), ts))

def _ftc2(pair='f=x^2,  F=x^3/3', a=0.0, b=1.0):
    f, F, _, _ = _PAIRS[pair]
    if b - a < 0.2:
        b = a + 0.2
    lo, hi = min(a, b), max(a, b)
    pad = 0.35 * (hi - lo) + 0.2
    xs = np.linspace(lo - pad, hi + pad, 600)
    # 1/x and ln need a positive window; clip safely.
    if 'ln x' in pair or '1/x' in pair:
        xs = xs[xs > 1e-3]

    area = _trap_area(f, a, b)
    exact = F(b) - F(a)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 6.8),
                                   sharex=True)
    # --- Top: f with the area from a to b shaded. ---
    ax1.plot(xs, f(xs), color=BLUE, lw=2.4, label='$f(x)$', zorder=3)
    fillx = np.linspace(a, b, 400)
    ax1.fill_between(fillx, 0, f(fillx), color=SHADEG, alpha=0.95,
                     zorder=1, label='$\\int_a^b f$')
    ax1.axhline(0, color=GRAY, lw=0.8)
    ax1.axvline(a, color=RED, lw=1.0, ls=':')
    ax1.axvline(b, color=RED, lw=1.0, ls=':')
    ax1.set_ylabel('$f(x)$')
    ax1.set_title(f'Area $\\int_a^b f$ = {area:.4f}', color=DARK)
    ax1.legend(loc='best', framealpha=0.9, fontsize=10)

    # --- Bottom: F with the endpoint values and their rise. ---
    ax2.plot(xs, F(xs), color=ORANGE, lw=2.4, label='$F(x)$',
             zorder=3)
    Fa, Fb = F(a), F(b)
    ax2.scatter([a, b], [Fa, Fb], s=80, color=RED, zorder=6)
    ax2.annotate(f'$F(a)$ = {Fa:.3f}', xy=(a, Fa),
                 xytext=(6, -14), textcoords='offset points',
                 color=RED, fontsize=10)
    ax2.annotate(f'$F(b)$ = {Fb:.3f}', xy=(b, Fb),
                 xytext=(6, 8), textcoords='offset points',
                 color=RED, fontsize=10)
    # The rise F(b) - F(a), drawn as a vertical gap at x = b.
    ax2.plot([b, b], [Fa, Fb], color=GREEN, lw=3.0, alpha=0.8,
             zorder=4, label='rise $F(b)-F(a)$')
    ax2.plot([a, b], [Fa, Fa], color=GREEN, lw=1.0, ls=':',
             zorder=2)
    ax2.axvline(a, color=RED, lw=1.0, ls=':')
    ax2.axvline(b, color=RED, lw=1.0, ls=':')
    ax2.set_xlabel('$x$')
    ax2.set_title(f'$F(b)-F(a)$ = {exact:.4f}   '
                  f'(matches the area)', color=DARK)
    ax2.legend(loc='best', framealpha=0.9, fontsize=10)
    plt.tight_layout()
    plt.show()

interact(_ftc2,
         pair=Dropdown(options=list(_PAIRS), description='(f, F)'),
         a=FloatSlider(value=0.0, min=-1.5, max=2.0, step=0.1,
                       description='a'),
         b=FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1,
                       description='b'));


interactive(children=(Dropdown(description='(f, F)', options=('f=x^2,  F=x^3/3', 'f=sin x,  F=-cos x', 'f=cos …

## 3. Integration by parts — the product rule, backward

The product rule integrated and rearranged is

$$\int u\,dv = uv - \int v\,du.$$

Use it when the integrand is a product and differentiating one factor makes the leftover integral *simpler*. Choose that factor to be $u$ (it gets differentiated) and the rest to be $dv$ (it gets integrated). Pick an example: the widget prints the $u,\,dv,\,v,\,du$ bookkeeping and the resulting $uv-\int v\,du$, then plots the integrand (blue) and its antiderivative (orange). Check visually that the orange curve's slope *is* the blue curve.

In [ ]:
# Each example: integrand, its antiderivative, and the parts table.
_PARTS = {
    'int x e^x dx': dict(
        integrand=lambda x: x*np.exp(x),
        antideriv=lambda x: (x - 1)*np.exp(x),
        u='x', dv='e^x dx', v='e^x', du='1 dx',
        result=r'x e^x - \int e^x\,dx = (x-1)e^x + C',
        xlim=(-3.0, 2.0)),
    'int x sin x dx': dict(
        integrand=lambda x: x*np.sin(x),
        antideriv=lambda x: np.sin(x) - x*np.cos(x),
        u='x', dv='sin x dx', v='-cos x', du='1 dx',
        result=r'-x\cos x - \int(-\cos x)\,dx = \sin x - x\cos x + C',
        xlim=(-2*np.pi, 2*np.pi)),
    'int x cos x dx': dict(
        integrand=lambda x: x*np.cos(x),
        antideriv=lambda x: np.cos(x) + x*np.sin(x),
        u='x', dv='cos x dx', v='sin x', du='1 dx',
        result=r'x\sin x - \int \sin x\,dx = \cos x + x\sin x + C',
        xlim=(-2*np.pi, 2*np.pi)),
    'int ln x dx': dict(
        integrand=lambda x: np.log(x),
        antideriv=lambda x: x*np.log(x) - x,
        u='ln x', dv='dx', v='x', du='(1/x) dx',
        result=r'x\ln x - \int x\cdot\tfrac1x\,dx = x\ln x - x + C',
        xlim=(0.05, 4.0)),
}

def _byparts(example='int x e^x dx'):
    d = _PARTS[example]
    # Render the bookkeeping table and the result as LaTeX/Markdown.
    table = (
        '| choose | differentiate | integrate |\n'
        '|---|---|---|\n'
        f'| $u = {d["u"]}$ | $du = {d["du"]}$ |  |\n'
        f'| $dv = {d["dv"]}$ |  | $v = {d["v"]}$ |\n'
    )
    display(Markdown(
        f'**Bookkeeping for $\\displaystyle\\int u\\,dv$:**\n\n'
        + table +
        f'\n$$\\int u\\,dv = uv - \\int v\\,du = {d["result"]}$$'
    ))

    lo, hi = d['xlim']
    xs = np.linspace(lo, hi, 600)
    fig, ax = plt.subplots()
    ax.plot(xs, d['integrand'](xs), color=BLUE, lw=2.4,
            label='integrand', zorder=3)
    ax.plot(xs, d['antideriv'](xs), color=ORANGE, lw=2.4,
            label='antiderivative (its slope = integrand)', zorder=3)
    ax.axhline(0, color=GRAY, lw=0.8)
    ax.set_xlim(lo, hi)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('Integration by parts: integrand vs antiderivative',
                 color=DARK)
    ax.legend(loc='best', framealpha=0.9, fontsize=10)
    plt.show()

interact(_byparts,
         example=Dropdown(options=list(_PARTS), description='example'));


## 4. The logarithm as an area, and finding $e$

The book's final act. We *define* the natural logarithm as the area under $1/t$:

$$\ln x \;:=\; \int_1^x \frac{1}{t}\,dt \quad (x > 0).$$

Drag $x$ and watch the green area accumulate; the title reports both the shaded area and $\ln x$ — they are the same number. The vertical green marker shows where the area first reaches exactly $1$: that place is the number **$e$**. (For $x<1$ the integral runs backward, so the area is *negative* — exactly $\ln x < 0$.) Every loan the book took on $e$, $\ln$, and $e^x$ is repaid right here — *(Figure 11.2)*.

In [ ]:
def _log_area(x=2.0):
    lo_t, hi_t = 0.25, 6.0
    ts = np.linspace(lo_t, hi_t, 800)
    fig, ax = plt.subplots()
    ax.plot(ts, 1.0/ts, color=BLUE, lw=2.4, label='$y = 1/t$',
            zorder=3)

    # Shade the signed area from 1 to x (green); area = ln x.
    if x >= 1:
        fx = np.linspace(1, x, 400)
    else:
        fx = np.linspace(x, 1, 400)
    ax.fill_between(fx, 0, 1.0/fx, color=SHADEG, alpha=0.95,
                    zorder=1, label='area $= \\ln x$')
    area = float(np.log(x))            # = int_1^x dt/t, exactly

    # Mark t = e, where the area from 1 first reaches exactly 1.
    e = np.e
    ax.axvline(e, color=GREEN, lw=2.0, ls='--', zorder=4)
    ax.annotate('$t = e$  (area here $= 1$)', xy=(e, 1.0/e),
                xytext=(e + 0.15, 0.62), color=GREEN, fontsize=11,
                arrowprops=dict(arrowstyle='->', color=GREEN))
    ax.scatter([e], [1.0/e], s=70, color=GREEN, zorder=6)
    # Mark the movable right edge x.
    ax.axvline(x, color=RED, lw=1.2, ls=':', zorder=4)
    ax.scatter([x], [1.0/x], s=70, color=RED, zorder=6)
    ax.axvline(1, color=GRAY, lw=1.0)
    ax.axhline(0, color=GRAY, lw=0.8)

    ax.set_xlim(lo_t, hi_t)
    ax.set_ylim(0, 4.2)
    ax.set_xlabel('$t$'); ax.set_ylabel('$1/t$')
    ax.set_title(f'x = {x:.3f}:   area = {area:+.4f}   '
                 f'=   ln x = {area:+.4f}', color=DARK)
    ax.legend(loc='upper right', framealpha=0.9, fontsize=10)
    plt.show()

interact(_log_area,
         x=FloatSlider(value=2.0, min=0.3, max=6.0, step=0.05,
                       description='right edge x'));


---

*Four windows onto one theorem. Widgets 1 and 2 are the two halves of the Fundamental Theorem — the area function's slope *is* the curve, so a definite integral is just $F(b)-F(a)$. Widget 3 is the product rule run backward. Widget 4 is the summit: $\ln$ defined as an area, $e$ located as the place that area hits $1$, and every promissory note the book ever wrote paid in full. From here, Chapter 12 lets the sums run forever.*